[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ScriptsRemote/Transcend_SummerSchool/blob/main/Day3_Remote_Sensing_Fundamentals_and_Indices/05_poopo_water_extent_timelapse_es.ipynb)

# Día 3 - Fundamentos de Teledetección e Índices Espectrales
## Sesión 5: Reducción de la lámina de agua del Lago Poopó (animación GIF)

| | |
|---|---|
| **Fecha** | 2026-10-14 (miércoles) |
| **Horario** | Material complementario |
| **Entidad** | UoM |
| **Instructores** | Christhian, Sebastian |

### Contenido

> Reconstrucción de **40 años** de la lámina de agua del **Lago Poopó** (Bolivia) con Landsat 5, 7, 8 y 9 en
> Google Earth Engine: colección multisensor, máscara de nubes, índice **MNDWI**, regla de detección de agua,
> compuestos anuales, serie temporal de área inundada y, al final, una **animación GIF** que muestra la
> reducción del lago año a año junto con su curva de área.

### Objetivos de aprendizaje

- [ ] Unir Landsat 5, 7, 8 y 9 en una sola colección, aunque las bandas tengan nombres diferentes
- [ ] Detectar agua con el **MNDWI** y una segunda condición en SWIR1 para no confundir agua con costras de sal
- [ ] Construir **compuestos anuales** (mediana) para una misma estación del año
- [ ] Calcular el **área de agua por año** (km²) y evaluar la calidad de cada año (cobertura válida)
- [ ] Producir un **GIF** con Earth Engine y otro, anotado, que combine el mapa con la serie temporal
- [ ] Mapear **dónde** se perdió el agua comparando dos periodos

### Datos necesarios

No se usan archivos locales. Del catálogo de Earth Engine:

- `LANDSAT/LT05/C02/T1_L2`, `LANDSAT/LE07/C02/T1_L2`, `LANDSAT/LC08/C02/T1_L2`, `LANDSAT/LC09/C02/T1_L2`
  (Colección 2, reflectancia de superficie de Nivel 2)
- `JRC/GSW1_4/GlobalSurfaceWater` (solo para delimitar el dominio histórico del lago, como en el notebook 1)

### Por qué esto importa

El Lago Poopó es el segundo lago más grande de Bolivia y uno de los sistemas más **someros** del altiplano:
pocos metros de profundidad bastan para que su superficie cambie miles de km² de un año a otro. En diciembre de
2015 se declaró prácticamente **seco**. En el notebook 1 vimos esa dinámica con el producto JRC (hasta 2021) y en
el notebook 3 con SWOT (desde 2023). Aquí la reconstruimos **nosotros mismos** a partir de las imágenes
Landsat, aplicando los pasos del notebook 4 (escala, nubes, índices, umbrales), y la convertimos en una
animación fácil de comunicar.

---

## 5.0 - Configuración

In [ ]:
# Librerías (descomente la línea de pip en un entorno nuevo, p. ej. Google Colab)
#%pip install -q earthengine-api geemap

import ee                                      # API de Google Earth Engine
import geemap                                  # mapas interactivos y descarga de GIF
import numpy as np                             # arreglos numéricos
import pandas as pd                            # tablas (DataFrames)
import matplotlib.pyplot as plt                # gráficos
from matplotlib.patches import Patch           # elementos de leyenda
from PIL import Image, ImageSequence           # leer y escribir GIF cuadro a cuadro
from IPython.display import Image as IPImage, display   # mostrar el GIF en el notebook

In [ ]:
# Autenticar e inicializar Earth Engine
# >>> REEMPLAZAR por el id de su propio proyecto de Cloud <<<
EE_PROJECT = "ee-christhianscunha"

ee.Authenticate()
ee.Initialize(project=EE_PROJECT)

---

## 5.1 - Área de estudio y dominio histórico del lago

Usamos la misma **ventana de análisis** del notebook 1: un rectángulo que contiene todo el Lago Poopó, al sur
de Oruro.

Dentro de esa ventana hay ríos, bofedales, salares y suelo desnudo. Para medir solo el lago, limitamos el
cálculo de área al **dominio histórico**: los píxeles donde el producto JRC detectó agua alguna vez
(`max_extent = 1`). Así evitamos contar como "agua" sombras o suelos húmedos lejos del lago.

> El dominio es una delimitación **operativa** para el ejercicio, no un límite oficial del lago.

In [ ]:
# Ventana de análisis [oeste, sur, este, norte] (la misma del notebook 1)
ROI = ee.Geometry.Rectangle([-67.45, -19.20, -66.65, -18.30])

# Dominio histórico: píxeles donde JRC detectó agua alguna vez (1984-2021)
jrc = ee.Image("JRC/GSW1_4/GlobalSurfaceWater")
dominio = jrc.select("max_extent").eq(1).selfMask().clip(ROI).rename("dominio")

# Área del dominio en km2 (área de cada píxel x 1 donde hay dominio)
area_dominio = (ee.Image.pixelArea().updateMask(dominio)
                  .reduceRegion(ee.Reducer.sum(), ROI, 30, maxPixels=1e10)
                  .get("area").getInfo() / 1e6)
print(f"Área del dominio histórico: {area_dominio:,.0f} km2")

Map = geemap.Map()
Map.add_basemap("HYBRID")
Map.centerObject(ROI, 9)
Map.addLayer(ROI, {"color": "red"}, "Ventana de análisis", True, 0.3)
Map.addLayer(dominio, {"palette": ["#ffd700"]}, "Dominio histórico (JRC max_extent)", True, 0.5)
Map

---

## 5.2 - Una colección Landsat de 40 años (5, 7, 8 y 9)

Para cubrir 1985-2025 necesitamos **cuatro satélites**:

| Satélite | Sensor | Periodo útil | Observación |
|---|---|---|---|
| Landsat 5 | TM | 1984 - 2011 | la base de la serie histórica |
| Landsat 7 | ETM+ | 1999 - hoy | desde 2003 tiene franjas sin datos (falla del SLC) |
| Landsat 8 | OLI | 2013 - hoy | |
| Landsat 9 | OLI-2 | 2021 - hoy | casi idéntico a Landsat 8 |

El problema: **las mismas longitudes de onda tienen nombres de banda diferentes**.

| Nombre que usaremos | Landsat 5 / 7 | Landsat 8 / 9 |
|---|---|---|
| Blue | SR_B1 | SR_B2 |
| Green | SR_B2 | SR_B3 |
| Red | SR_B3 | SR_B4 |
| NIR | SR_B4 | SR_B5 |
| SWIR1 | SR_B5 | SR_B6 |
| SWIR2 | SR_B7 | SR_B7 |

La solución es la misma del notebook 4: una función que **escala, enmascara nubes y renombra**, ahora con la
lista de bandas como parámetro. Después de renombrar, las cuatro colecciones "hablan el mismo idioma" y
se pueden unir con `merge`. El factor de escala y los bits de `QA_PIXEL` son los mismos para los cuatro
satélites en la Colección 2.

### ¿Qué época del año?

El lago cambia **dentro** de cada año: crece con las lluvias (diciembre - marzo) y baja en la época seca.
Para comparar años de forma justa usamos siempre **la misma ventana de meses**. Elegimos **abril - julio**: justo
después de las lluvias (el lago todavía refleja el aporte del año) y con pocas nubes.

In [ ]:
# Parámetros de la serie
ANIO_INICIO, ANIO_FIN = 1985, 2025
MES_INICIO, MES_FIN = 4, 7          # abril a julio
MAX_NUBES = 40                      # % máximo de nubes por escena

# Nombres originales de las bandas ópticas en cada sensor, y los nombres comunes
BANDAS_L57 = ["SR_B1", "SR_B2", "SR_B3", "SR_B4", "SR_B5", "SR_B7"]   # Landsat 5 y 7
BANDAS_L89 = ["SR_B2", "SR_B3", "SR_B4", "SR_B5", "SR_B6", "SR_B7"]   # Landsat 8 y 9
NOMBRES = ["Blue", "Green", "Red", "NIR", "SWIR1", "SWIR2"]


def preparar(image, bandas):
    # Máscara de nubes: bits 1 (nube dilatada), 2 (cirros), 3 (nube) y 4 (sombra)
    qa = image.select("QA_PIXEL")
    bits_nube = (1 << 1) | (1 << 2) | (1 << 3) | (1 << 4)
    despejado = qa.bitwiseAnd(bits_nube).eq(0)

    # Factor de escala de la Colección 2 y nombres comunes
    optico = image.select(bandas).multiply(0.0000275).add(-0.2).rename(NOMBRES)

    # Conservar fecha, nubes y satélite de la imagen original
    return ee.Image(optico.updateMask(despejado).copyProperties(
        image, ["system:time_start", "CLOUD_COVER", "SPACECRAFT_ID"]))


def cargar(coleccion_id, bandas):
    return (ee.ImageCollection(coleccion_id)
              .filterBounds(ROI)
              .filter(ee.Filter.calendarRange(ANIO_INICIO, ANIO_FIN, "year"))
              .filter(ee.Filter.calendarRange(MES_INICIO, MES_FIN, "month"))
              .filter(ee.Filter.lt("CLOUD_COVER", MAX_NUBES))
              .map(lambda img: preparar(img, bandas)))


# Las cuatro colecciones, ya con los mismos nombres de banda, en una sola
landsat = (cargar("LANDSAT/LT05/C02/T1_L2", BANDAS_L57)
             .merge(cargar("LANDSAT/LE07/C02/T1_L2", BANDAS_L57))
             .merge(cargar("LANDSAT/LC08/C02/T1_L2", BANDAS_L89))
             .merge(cargar("LANDSAT/LC09/C02/T1_L2", BANDAS_L89)))

print("Número de escenas:", landsat.size().getInfo())
print("Bandas:", landsat.first().bandNames().getInfo())

### ¿Cuántas escenas tenemos por año y por satélite?

Antes de calcular nada, conviene saber **con qué datos contamos** cada año. Un año con pocas escenas dará un
compuesto menos confiable.

In [ ]:
# Fecha y satélite de cada escena
escenas = pd.DataFrame({
    "fecha": pd.to_datetime(landsat.aggregate_array("system:time_start").getInfo(), unit="ms"),
    "satelite": landsat.aggregate_array("SPACECRAFT_ID").getInfo(),
})
escenas["anio"] = escenas["fecha"].dt.year

# Tabla: una fila por año, una columna por satélite
por_anio = escenas.groupby(["anio", "satelite"]).size().unstack(fill_value=0)

ax = por_anio.plot(kind="bar", stacked=True, figsize=(12, 4), width=0.85,
                   color=["#8c510a", "#d8b365", "#5ab4ac", "#01665e"][: por_anio.shape[1]])
ax.set_title(f"Escenas Landsat por año (meses {MES_INICIO}-{MES_FIN}, nubes < {MAX_NUBES}%)")
ax.set_xlabel("Año")
ax.set_ylabel("Número de escenas")
ax.legend(title="Satélite", frameon=False)
ax.spines[["top", "right"]].set_visible(False)
plt.show()

por_anio.T

---

## 5.3 - Detectar agua: MNDWI + una segunda condición

En el notebook 4 usamos el **NDWI** (Green y NIR). Para lagos someros y turbios como el Poopó es más robusto el
**MNDWI** (Xu, 2006), que reemplaza el NIR por el **SWIR1**:

$$MNDWI = \frac{Green - SWIR1}{Green + SWIR1}$$

El agua absorbe casi toda la energía en el SWIR, así que el MNDWI es positivo sobre agua, incluso con
sedimentos.

**Cuidado con la sal.** Cuando el Poopó se seca deja **costras salinas** muy brillantes en todas las bandas. Sobre
la sal, Green es apenas mayor que SWIR1 y el MNDWI puede quedar **ligeramente positivo** - la sal se
"confundiría" con agua. Por eso añadimos una segunda condición: el agua es **oscura en el SWIR1**.

| Condición | Por qué |
|---|---|
| `MNDWI > 0` | más reflectancia en el verde que en el SWIR: agua |
| `SWIR1 < 0.10` | el agua absorbe el SWIR; la sal y el suelo seco no |

Las dos condiciones se combinan con `.And()`. Ambos umbrales son un **punto de partida**: verifíquelos en el mapa.

In [ ]:
UMBRAL_MNDWI = 0.0
UMBRAL_SWIR1 = 0.10


def agregar_agua(image):
    # MNDWI como nueva banda
    mndwi = image.normalizedDifference(["Green", "SWIR1"]).rename("MNDWI")
    # Agua = MNDWI alto Y SWIR1 oscuro (1 = agua, 0 = no agua)
    agua = (mndwi.gt(UMBRAL_MNDWI)
                 .And(image.select("SWIR1").lt(UMBRAL_SWIR1))
                 .rename("agua"))
    return image.addBands([mndwi, agua])

---

## 5.4 - Compuestos anuales

Para cada año calculamos la **mediana** de todas las escenas de abril a julio. La mediana:

- rellena los huecos que dejan las nubes (y buena parte de las franjas de Landsat 7);
- descarta valores extremos (una nube no detectada, una sombra);
- da **una imagen por año**, que será **un cuadro** de la animación.

Construimos la lista de compuestos con una *list comprehension* de Python: un compuesto por cada año que tiene
al menos una escena. Cada compuesto guarda su año (`anio`) y el número de escenas usadas (`n_imagenes`).

In [ ]:
# Años que tienen al menos una escena (de la tabla anterior)
anios = sorted(int(a) for a in escenas["anio"].unique())


def compuesto_anual(anio):
    imgs = landsat.filter(ee.Filter.calendarRange(anio, anio, "year"))
    comp = agregar_agua(imgs.median().clip(ROI))
    return comp.set({
        "anio": anio,
        "n_imagenes": imgs.size(),
        "system:time_start": ee.Date.fromYMD(anio, MES_INICIO, 1).millis(),
    })


# Una imagen por año, en orden cronológico
anuales = ee.ImageCollection([compuesto_anual(a) for a in anios])

print(f"{len(anios)} compuestos anuales: {anios[0]} - {anios[-1]}")
print("Bandas:", anuales.first().bandNames().getInfo())

### Dos años contrastantes en el mapa

Comparemos un año con el lago **lleno** y uno con el lago **seco**. Usamos falso color
**SWIR1 - NIR - Green**: el agua aparece en azul oscuro / negro, la sal en blanco - celeste, la
vegetación en verde y el suelo desnudo en tonos marrones.

Use el control de capas para encender y apagar cada año y la máscara de agua.

In [ ]:
ANIO_A = 1986   # crecida histórica
ANIO_B = 2016   # el lago después de la desecación de 2015

vis_falso = {"bands": ["SWIR1", "NIR", "Green"], "min": 0.0, "max": 0.4}   # agua oscura, sal clara
vis_rgb = {"bands": ["Red", "Green", "Blue"], "min": 0.0, "max": 0.3}      # color natural
vis_agua = {"palette": ["#00bfff"]}


def compuesto_de(anio):
    return ee.Image(anuales.filter(ee.Filter.eq("anio", anio)).first())


img_a, img_b = compuesto_de(ANIO_A), compuesto_de(ANIO_B)

Map_anios = geemap.Map()
Map_anios.centerObject(ROI, 9)
Map_anios.addLayer(img_a, vis_rgb, f"{ANIO_A} - color natural", False)
Map_anios.addLayer(img_b, vis_rgb, f"{ANIO_B} - color natural", False)
Map_anios.addLayer(img_a, vis_falso, f"{ANIO_A} - falso color")
Map_anios.addLayer(img_b, vis_falso, f"{ANIO_B} - falso color")
Map_anios.addLayer(img_a.select("agua").selfMask(), vis_agua, f"{ANIO_A} - agua", False)
Map_anios.addLayer(img_b.select("agua").selfMask(), vis_agua, f"{ANIO_B} - agua", False)
Map_anios

---

## 5.5 - Área de agua por año

Igual que en el notebook 4: `ee.Image.pixelArea()` por la máscara (1/0) y una **suma** sobre la región da
el área de agua. Ahora lo hacemos para **cada compuesto** con `.map()` y traemos todo en **una sola** llamada
a `getInfo()`.

Calculamos también la **cobertura válida**: qué porcentaje del dominio tiene datos en el compuesto. Si un
año quedó con muchos píxeles sin datos (nubes persistentes, pocas escenas), su área estará **subestimada**.

> Usamos `ESCALA = 60` m para que el cálculo de 40 años termine rápido. Con 30 m el resultado casi no cambia
> para un lago de cientos de km², pero tarda más.

In [ ]:
ESCALA = 60   # m


def area_agua(img):
    pixel = ee.Image.pixelArea()
    agua = img.select("agua").unmask(0)            # sin datos -> no agua
    valido = img.select("Green").mask().gt(0)      # 1 = el compuesto tiene dato
    stats = (ee.Image.cat([
                 agua.multiply(pixel).rename("agua_m2"),
                 valido.multiply(pixel).rename("valido_m2"),
                 pixel.rename("total_m2"),
             ])
             .updateMask(dominio)                  # solo dentro del dominio histórico
             .reduceRegion(reducer=ee.Reducer.sum(), geometry=ROI, scale=ESCALA,
                           maxPixels=1e10, tileScale=4))
    return ee.Feature(None, {
        "anio": img.get("anio"),
        "n_imagenes": img.get("n_imagenes"),
        "agua_km2": ee.Number(stats.get("agua_m2")).divide(1e6),
        "valido_pct": ee.Number(stats.get("valido_m2")).divide(stats.get("total_m2")).multiply(100),
    })


# Un Feature por año -> una sola llamada a getInfo -> DataFrame
tabla = ee.FeatureCollection(anuales.map(area_agua)).getInfo()
df = (pd.DataFrame([f["properties"] for f in tabla["features"]])
        .sort_values("anio").reset_index(drop=True)
        [["anio", "n_imagenes", "agua_km2", "valido_pct"]]
        .round({"agua_km2": 1, "valido_pct": 1}))

# Años con poca cobertura válida: interpretar con cuidado
COBERTURA_MIN = 80   # %
df["confiable"] = df["valido_pct"] >= COBERTURA_MIN
df

In [ ]:
ok, dudoso = df[df["confiable"]], df[~df["confiable"]]
i_max, i_min = df.loc[ok.index, "agua_km2"].idxmax(), df.loc[ok.index, "agua_km2"].idxmin()

fig, ax = plt.subplots(figsize=(12, 4.5))
ax.fill_between(df["anio"], df["agua_km2"], color="#2c7bb6", alpha=0.15)
ax.plot(df["anio"], df["agua_km2"], color="#2c7bb6", linewidth=2, label="área de agua")
ax.scatter(ok["anio"], ok["agua_km2"], color="#2c7bb6", zorder=3)
ax.scatter(dudoso["anio"], dudoso["agua_km2"], facecolor="white", edgecolor="#2c7bb6", zorder=3,
           label=f"cobertura válida < {COBERTURA_MIN}%")

# Marcar el máximo y el mínimo de la serie
for i, color, txt in [(i_max, "#1a9641", "máximo"), (i_min, "#d7191c", "mínimo")]:
    ax.scatter(df.loc[i, "anio"], df.loc[i, "agua_km2"], s=120, color=color, zorder=4)
    ax.annotate(f"{txt}: {df.loc[i, 'agua_km2']:,.0f} km² ({df.loc[i, 'anio']})",
                (df.loc[i, "anio"], df.loc[i, "agua_km2"]), textcoords="offset points",
                xytext=(10, 10), color=color, fontweight="bold")

ax.set_title(f"Lago Poopó - área de agua (Landsat, meses {MES_INICIO}-{MES_FIN})")
ax.set_xlabel("Año")
ax.set_ylabel("Área de agua (km²)")
ax.set_ylim(bottom=0)
ax.legend(frameon=False, loc="upper right")
ax.spines[["top", "right"]].set_visible(False)
plt.show()

### ¿Cuánto se redujo la lámina de agua?

Resumimos la serie en pocos números: el máximo, el mínimo, el último año y la **reducción relativa** respecto al
máximo. Solo usamos los años con buena cobertura.

In [ ]:
a_max, a_min = df.loc[i_max], df.loc[i_min]
ultimo = ok.iloc[-1]

resumen = pd.DataFrame({
    "año": [int(a_max["anio"]), int(a_min["anio"]), int(ultimo["anio"])],
    "área (km2)": [a_max["agua_km2"], a_min["agua_km2"], ultimo["agua_km2"]],
    "% del máximo": [100.0,
                     100 * a_min["agua_km2"] / a_max["agua_km2"],
                     100 * ultimo["agua_km2"] / a_max["agua_km2"]],
}, index=["máximo", "mínimo", "último año"]).round(1)

print(f"Reducción máxima: {a_max['agua_km2'] - a_min['agua_km2']:,.0f} km2 "
      f"({100 * (1 - a_min['agua_km2'] / a_max['agua_km2']):.0f}% del área máxima)")
resumen

---

## 5.6 - Primer GIF: la animación directamente desde Earth Engine

Earth Engine puede convertir una `ImageCollection` en una animación con `getVideoThumbURL`. Cada imagen
de la colección es **un cuadro**, en el orden de la colección. Requisitos:

- cada cuadro debe ser una imagen **RGB de 8 bits** - eso es exactamente lo que devuelve `visualize()`;
- hay que indicar la región, el tamaño en píxeles (`dimensions`) y la velocidad (`framesPerSecond`).

`geemap.download_ee_video` pide la URL y guarda el GIF en el disco.

Creamos una función `fotograma` que dibuja el compuesto en falso color y, opcionalmente, pinta encima la
máscara de agua en celeste con `blend()`.

In [ ]:
def fotograma(img, con_agua=False):
    # Fondo en falso color (RGB de 8 bits)
    cuadro = img.visualize(**vis_falso)
    if con_agua:
        # Máscara de agua semitransparente encima del fondo
        agua = img.select("agua").selfMask().visualize(palette=["00bfff"], opacity=0.6)
        cuadro = cuadro.blend(agua)
    return cuadro


video_args = {
    "region": ROI,
    "dimensions": 600,          # lado mayor del GIF, en píxeles
    "framesPerSecond": 3,
    "crs": "EPSG:3857",
}

# Cuadros solo con la imagen satelital: lo que "ve" el satélite
cuadros_sat = anuales.map(lambda img: fotograma(img, con_agua=False))

GIF_SAT = "poopo_falso_color.gif"
geemap.download_ee_video(cuadros_sat, video_args, GIF_SAT)
display(IPImage(filename=GIF_SAT))

El GIF de Earth Engine no indica **qué año** muestra cada cuadro. En la siguiente sección lo
completamos.

---

## 5.7 - GIF final: mapa + serie temporal

Ahora descargamos los cuadros **con la máscara de agua** y, cuadro a cuadro, armamos una figura con
`matplotlib`:

- a la izquierda, el mapa del año, con su título;
- a la derecha, la curva de área: en gris la serie completa, en azul lo recorrido hasta ese año y un
  punto rojo en el año actual.

Cada figura se convierte en una imagen (`Image.fromarray`) y al final todas se guardan como un GIF con
`Image.save(..., save_all=True)`. El último cuadro queda más tiempo en pantalla.

In [ ]:
# 1) Cuadros con la máscara de agua, desde Earth Engine
cuadros_agua = anuales.map(lambda img: fotograma(img, con_agua=True))

GIF_AGUA = "poopo_agua.gif"
geemap.download_ee_video(cuadros_agua, video_args, GIF_AGUA)


# 2) Leer los cuadros del GIF como imágenes RGB
def leer_cuadros(ruta):
    with Image.open(ruta) as gif:
        return [cuadro.convert("RGB") for cuadro in ImageSequence.Iterator(gif)]


cuadros = leer_cuadros(GIF_AGUA)
print(f"{len(cuadros)} cuadros de {cuadros[0].size[0]} x {cuadros[0].size[1]} píxeles")
assert len(cuadros) == len(df), "El número de cuadros no coincide con el número de años"

In [ ]:
def gif_mapa_y_serie(cuadros, df, ruta_salida, ms_por_cuadro=500, ms_ultimo=2500, dpi=80):
    a_max = df["agua_km2"].max()
    leyenda = [Patch(color="#00bfff", label="agua detectada (MNDWI)")]
    salida = []

    for i, cuadro in enumerate(cuadros):
        fila = df.iloc[i]
        fig, (ax_map, ax_ts) = plt.subplots(1, 2, figsize=(12, 5.2), dpi=dpi,
                                            gridspec_kw={"width_ratios": [1, 1.3]})

        # --- mapa del año ---
        ax_map.imshow(cuadro)
        ax_map.set_title(f"Lago Poopó - {int(fila['anio'])}", fontsize=16, fontweight="bold")
        ax_map.axis("off")
        ax_map.legend(handles=leyenda, loc="lower left", fontsize=9, framealpha=0.8)

        # --- serie temporal ---
        hasta = df.iloc[: i + 1]
        ax_ts.plot(df["anio"], df["agua_km2"], color="lightgrey", linewidth=1.5)
        ax_ts.fill_between(hasta["anio"], hasta["agua_km2"], color="#2c7bb6", alpha=0.15)
        ax_ts.plot(hasta["anio"], hasta["agua_km2"], color="#2c7bb6", linewidth=2.5)
        ax_ts.scatter([fila["anio"]], [fila["agua_km2"]], s=90, color="#d7191c", zorder=3)
        texto = f"{fila['agua_km2']:,.0f} km²  ({100 * fila['agua_km2'] / a_max:.0f}% del máximo)"
        if not fila["confiable"]:
            texto += "\ncobertura válida baja"
        ax_ts.text(0.98, 0.97, texto, transform=ax_ts.transAxes, va="top", ha="right",
                   fontsize=13, fontweight="bold", color="#d7191c",
                   bbox={"facecolor": "white", "edgecolor": "none", "alpha": 0.85})
        ax_ts.set_xlim(df["anio"].min() - 1, df["anio"].max() + 1)
        ax_ts.set_ylim(0, a_max * 1.15)
        ax_ts.set_xlabel("Año")
        ax_ts.set_ylabel("Área de agua (km²)")
        ax_ts.set_title(f"Área de agua (meses {MES_INICIO}-{MES_FIN})")
        ax_ts.spines[["top", "right"]].set_visible(False)

        fig.text(0.99, 0.01, "Landsat 5/7/8/9 (USGS/NASA) - Google Earth Engine - MNDWI",
                 ha="right", fontsize=8, color="grey")
        fig.tight_layout(rect=[0, 0.03, 1, 1])

        # Figura -> imagen RGB
        fig.canvas.draw()
        salida.append(Image.fromarray(np.asarray(fig.canvas.buffer_rgba())[..., :3]))
        plt.close(fig)

    # Guardar todos los cuadros como un GIF que se repite (loop=0)
    duraciones = [ms_por_cuadro] * (len(salida) - 1) + [ms_ultimo]
    salida[0].save(ruta_salida, save_all=True, append_images=salida[1:],
                   duration=duraciones, loop=0)
    return ruta_salida


GIF_FINAL = "poopo_reduccion_lamina_agua.gif"
gif_mapa_y_serie(cuadros, df, GIF_FINAL)
display(IPImage(filename=GIF_FINAL))

**Para descargar el GIF** en Google Colab: abra el panel de archivos (icono de carpeta a la izquierda), haga clic
derecho sobre `poopo_reduccion_lamina_agua.gif` y elija *Descargar*, o ejecute la celda siguiente.

In [ ]:
# Solo en Google Colab: descargar el GIF final a su computador
try:
    from google.colab import files
    files.download(GIF_FINAL)
except ImportError:
    print(f"El GIF está guardado en: {GIF_FINAL}")

---

## 5.8 - ¿Dónde se perdió el agua?

La animación muestra **cuándo**. Para ver **dónde**, calculamos la **frecuencia de agua** en dos periodos:
la media de la banda `agua` (1/0) de todos los compuestos del periodo. Un valor de 1 significa "agua todos
los años"; 0, "nunca agua".

La diferencia **reciente - antiguo** muestra en rojo las zonas que perdieron agua y en azul las que ganaron.
Es la misma idea de `change_abs` de JRC (notebook 1), pero calculada por nosotros.

In [ ]:
PERIODO_ANTIGUO = (1985, 1999)
PERIODO_RECIENTE = (2011, 2025)


def frecuencia_agua(inicio, fin):
    sel = anuales.filter(ee.Filter.rangeContains("anio", inicio, fin))
    return sel.select("agua").mean().updateMask(dominio)


frec_antigua = frecuencia_agua(*PERIODO_ANTIGUO)
frec_reciente = frecuencia_agua(*PERIODO_RECIENTE)
cambio = frec_reciente.subtract(frec_antigua).rename("cambio")

vis_frec = {"min": 0, "max": 1, "palette": ["#f7fbff", "#6baed6", "#08306b"]}
vis_cambio = {"min": -1, "max": 1, "palette": ["#b2182b", "#f7f7f7", "#2166ac"]}

Map_cambio = geemap.Map()
Map_cambio.add_basemap("HYBRID")
Map_cambio.centerObject(ROI, 9)
Map_cambio.addLayer(frec_antigua, vis_frec, f"Frecuencia de agua {PERIODO_ANTIGUO[0]}-{PERIODO_ANTIGUO[1]}", False)
Map_cambio.addLayer(frec_reciente, vis_frec, f"Frecuencia de agua {PERIODO_RECIENTE[0]}-{PERIODO_RECIENTE[1]}", False)
Map_cambio.addLayer(cambio, vis_cambio, "Cambio (reciente - antiguo)")
Map_cambio.add_colorbar(vis_cambio, label="Cambio en la frecuencia de agua (rojo = pérdida)")
Map_cambio

---

## 5.9 - Limitaciones

| Limitación | Efecto | Qué hacer |
|---|---|---|
| Umbrales fijos (`MNDWI > 0`, `SWIR1 < 0.10`) | agua muy turbia o muy somera puede quedar fuera; suelo salino húmedo puede entrar | probar otros valores y verificar visualmente (ejercicio 2) |
| Sensores diferentes (TM, ETM+, OLI) | pequeñas diferencias de reflectancia entre satélites | para índices y umbrales el efecto es pequeño; para análisis finos, aplicar una armonización (Roy et al., 2016) |
| Franjas de Landsat 7 (desde 2003) | huecos en años con pocas escenas de otros satélites (p. ej. 2012) | revisar la `valido_pct` de esos años |
| Una sola ventana de meses | no captura la dinámica **dentro** del año | repetir con otros meses (ejercicio 3) |
| Dominio JRC (1984-2021) | agua nueva fuera del dominio histórico no se cuenta | poco probable en el Poopó, pero vale la pena revisar el mapa |
| Escala de 60 m | bordes del lago menos precisos | usar `ESCALA = 30` para el resultado final |

---

## 5.10 - Resumen

| Paso | Qué hicimos | Herramienta |
|---|---|---|
| 1 | Dominio histórico del lago | `JRC/GSW1_4`, `max_extent`, `selfMask` |
| 2 | Colección Landsat 5 + 7 + 8 + 9 | `select(...).rename(...)`, `merge`, `calendarRange` |
| 3 | Detección de agua | `normalizedDifference` (MNDWI), `gt`, `lt`, `And` |
| 4 | Una imagen por año | `median()`, list comprehension, `ee.ImageCollection([...])` |
| 5 | Área de agua y calidad por año | `pixelArea`, `reduceRegion`, `.map()`, `getInfo()` |
| 6 | GIF desde Earth Engine | `visualize`, `blend`, `getVideoThumbURL` (`geemap.download_ee_video`) |
| 7 | GIF anotado con serie temporal | `PIL.ImageSequence`, `matplotlib`, `Image.save(save_all=True)` |
| 8 | Dónde se perdió el agua | `mean()` por periodo y diferencia |

---

## 5.11 - Ejercicio

1. **Compare con JRC.** Para 1985-2021, compare el área de este notebook con el área anual de JRC calculada en el
   notebook 1. ¿En qué años difieren más? ¿Por qué? (piense en la ventana de meses y en la sal).
2. **Cambie los umbrales.** Pruebe `UMBRAL_MNDWI = 0.2` y `UMBRAL_SWIR1 = 0.05`, y luego quite la condición del
   SWIR1. ¿Qué pasa con el área en los años secos (2016)? Mire el mapa de 5.4 para explicar la diferencia.
3. **Otra estación.** Repita con `MES_INICIO, MES_FIN = 9, 11` (final de la época seca). ¿El lago es más
   pequeño? ¿Cambia el año del mínimo?
4. **Otro lago.** Cambie `ROI` para el **Lago Uru Uru** (al norte de Poopó) o para el **Lago Titicaca menor** y
   genere su GIF. ¿Ve la misma tendencia?
5. **Mejore el GIF.** Cambie `framesPerSecond`, `dimensions` o la visualización (por ejemplo color natural
   `vis_rgb`) y compare cuál comunica mejor la reducción del lago.

**Entregable**: el GIF final de su lago y un párrafo corto que responda: ¿cuándo y cuánto se redujo la lámina de
agua, y qué tan confiable es cada año de la serie?

---

## Referencias

**Índices y método**

- Xu, H. (2006). Modification of normalised difference water index (NDWI) to enhance open water features in
  remotely sensed imagery. *International Journal of Remote Sensing*, 27(14), 3025-3033. (MNDWI)
- McFeeters, S.K. (1996). The use of the Normalized Difference Water Index (NDWI) in the delineation of open
  water features. *International Journal of Remote Sensing*, 17(7), 1425-1432. (NDWI)
- Pekel, J.-F., Cottam, A., Gorelick, N., Belward, A.S. (2016). High-resolution mapping of global surface water
  and its long-term changes. *Nature*, 540, 418-422. (JRC Global Surface Water)
- Roy, D.P. et al. (2016). Characterization of Landsat-7 to Landsat-8 reflective wavelength and normalized
  difference vegetation index continuity. *Remote Sensing of Environment*, 185, 57-70. (armonización entre sensores)

**Lago Poopó**

- Satgé, F. et al. (2017). Role of climate variability and human activity on Poopó Lake droughts between 1990
  and 2015 assessed using remote sensing data. *Remote Sensing*, 9(3), 218.

**Earth Engine y Landsat**

- Landsat 5 C2 L2: <https://developers.google.com/earth-engine/datasets/catalog/LANDSAT_LT05_C02_T1_L2>
- Landsat 7 C2 L2: <https://developers.google.com/earth-engine/datasets/catalog/LANDSAT_LE07_C02_T1_L2>
- Landsat 8 C2 L2: <https://developers.google.com/earth-engine/datasets/catalog/LANDSAT_LC08_C02_T1_L2>
- Landsat 9 C2 L2: <https://developers.google.com/earth-engine/datasets/catalog/LANDSAT_LC09_C02_T1_L2>
- JRC Global Surface Water: <https://developers.google.com/earth-engine/datasets/catalog/JRC_GSW1_4_GlobalSurfaceWater>
- Animaciones (`getVideoThumbURL`): <https://developers.google.com/earth-engine/guides/ic_visualization>
- geemap: <https://geemap.org>

**Libro (acceso abierto)**

- Cardille, J.A., Crowley, M.A., Saah, D., Clinton, N.E. (eds.) (2023). *Cloud-Based Remote Sensing with Google
  Earth Engine: Fundamentals and Applications*. Springer. <https://www.eefabook.org/>